# MV and LV power flow with SimPT-Power
Two distribution-level examples from the open bundle:
1. **MV**: an urban 10-kV cable network (MV-URBAN-CABLE-V1) behind one 60 kV/MV root transformer, solved with pandapower at the public peak-load proxy.
2. **LV**: a geographic four-wire (ABCN) LV network (LV-GEO-A) of one secondary substation (PTD), solved with the backward/forward sweep used to build the database; the result matches the stored one.
Pinned version: **pandapower 3.5.2**.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("pandapower") is None:  # on Kaggle: enable Internet in the notebook settings
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandapower==3.5.2"], check=True)
import pandapower as pp
print("pandapower", pp.__version__)

In [ ]:
import os, json
from pathlib import Path
def find_data():
    """Locate the SimPT-Power open bundle: $SIMPT_DATA, a Kaggle input dataset, or the local build folder."""
    cands = [os.environ.get("SIMPT_DATA", "")]
    cands += [str(p.parent) for p in Path("/kaggle/input").rglob("bundle_manifest.json")] if Path("/kaggle/input").exists() else []
    cands += [f"{pre}output/simpt_power_release/open_bundle_{tag}" for tag in ("r3", "r2") for pre in ("../", "")]
    for c in cands:
        if c and (Path(c) / "bundle_manifest.json").exists():
            return Path(c)
    raise FileNotFoundError("SimPT-Power bundle not found; set SIMPT_DATA")
DATA = find_data()
manifest = json.loads((DATA / "bundle_manifest.json").read_text())
print(DATA, "|", manifest["bundle_of"])

## 1. An urban MV cable network
Root `MVROOT:00298` (60/10 kV, Lisboa) feeds 181 PTDs through inferred underground cable feeders (207 segments, 33.7 km); 26 normally open segments tie neighbouring feeders.
The 60-kV source voltage is taken from the solved HV core at the national peak (15 Jan 2026 12:15 UTC). PTD loads are the public peak-load proxy (midpoint of each PTD's published utilisation band) at power factor 0.97.

In [ ]:
import math, duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
ROOT = "MVROOT:00298"
q = lambda sql: duckdb.sql(sql.replace("@", str(DATA) + "/")).df()
seg = q(f"select * from read_parquet('@mv/urban_cable_segments.parquet') where root = '{ROOT}'")
tr = q(f"select * from read_parquet('@mv/root_transformers.parquet') where mv_root_bus = '{ROOT}'").iloc[0]
ptd = q(f"select ptd_code, peak_load_proxy_mva, capacity_kva from read_parquet('@mv/ptd_connections.parquet') where mv_root_bus = '{ROOT}'")
vm_hv = q(f"select vm_pu from read_parquet('@hv/peak_20260115_1215_bus_voltage.parquet') where bus_id = '{tr.hv_bus}'").vm_pu.iloc[0]
print(f"{len(seg)} cable segments ({seg.normally_open.sum()} normally open), {seg.feeder.nunique()} feeders, {seg.length_km.sum():.1f} km")
print(f"root transformer {tr.hv_kv:.0f}/{tr.lv_kv:.0f} kV, {tr.sn_mva:.0f} MVA; {len(ptd)} PTDs; HV bus {tr.hv_bus} at {vm_hv:.4f} p.u.")
seg[["designation", "r_ohm_per_km", "x_ohm_per_km", "max_i_ka", "evidence_status"]].drop_duplicates()

In [ ]:
net = pp.create_empty_network()
hv = pp.create_bus(net, tr.hv_kv, name=tr.hv_bus)
pp.create_ext_grid(net, hv, vm_pu=vm_hv)
bus = {}
def b(name):
    if name not in bus:
        bus[name] = pp.create_bus(net, float(seg.kv.iloc[0]), name=name)
    return bus[name]
root = b(f"ROOTANCHOR:{ROOT}")
pp.create_transformer_from_parameters(net, hv, root, sn_mva=tr.sn_mva, vn_hv_kv=tr.hv_kv, vn_lv_kv=tr.lv_kv,
                                      vk_percent=tr.vk_percent, vkr_percent=tr.vkr_percent, pfe_kw=0, i0_percent=0)
for r in seg.itertuples():
    pp.create_line_from_parameters(net, b(r.from_node), b(r.to_node), length_km=r.length_km, r_ohm_per_km=r.r_ohm_per_km,
                                   x_ohm_per_km=r.x_ohm_per_km, c_nf_per_km=r.c_nf_per_km, max_i_ka=r.max_i_ka,
                                   in_service=not r.normally_open, name=r.segment_id)
tanphi = math.tan(math.acos(0.97))
for r in ptd.itertuples():
    p = r.peak_load_proxy_mva * 0.97
    pp.create_load(net, bus[r.ptd_code], p_mw=p, q_mvar=p * tanphi, name=r.ptd_code)
pp.runpp(net, numba=False)
print(f"converged {net.converged}: load {net.res_load.p_mw.sum():.1f} MW, min voltage {net.res_bus.vm_pu[net.bus.vn_kv < 60].min():.4f} p.u., "
      f"max cable loading {net.res_line.loading_percent.max():.1f}%, transformer loading {net.res_trafo.loading_percent.iloc[0]:.1f}%")

Voltage along each feeder (distance from the root) and the heaviest-loaded cable sections.

In [ ]:
import networkx as nx
g = nx.Graph()
for i, r in net.line[net.line.in_service].iterrows():
    g.add_edge(r.from_bus, r.to_bus, w=r.length_km)
dist = nx.single_source_dijkstra_path_length(g, root, weight="w")
d = pd.DataFrame({"km": pd.Series(dist), "vm_pu": net.res_bus.vm_pu})
d = d.dropna()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.scatter(d.km, d.vm_pu, s=8); ax.set_xlabel("distance from root (km)"); ax.set_ylabel("voltage (p.u.)")
ax.set_title(f"{ROOT}: MV bus voltages at the peak-load proxy"); plt.show()
net.res_line.join(net.line[["name", "length_km"]]).sort_values("loading_percent", ascending=False)[["name", "length_km", "loading_percent"]].head()

## 2. A geographic four-wire LV network
PTD `1106D1002700` (Lisboa, prefabricated cabin, 400 kVA) has 301 nodes: a short pole network and 5.3 km of street-routed underground cable (LXAV 3x185+95), with two parallel circuits added where the 0.9-p.u. screen required it (`refined_parallel`).
Each branch carries phases A, B, C and an explicit neutral N. The sweep of eq. (2) of the paper: currents are accumulated from the leaves, then voltages are updated from the transformer, until the change is below 1e-6 V.
Three inputs come from the full DuckDB release (tables named in comments) and are written here as constants: the PTD's phase shares, its design transformer rating and the 4x4 mutual-impedance proxy.

In [ ]:
from collections import defaultdict, deque
VLN = 400 / math.sqrt(3)
SRC = np.array([VLN * np.exp(1j * a) for a in (0, -2 * math.pi / 3, 2 * math.pi / 3)] + [0j])
MUTUAL = 0.05 + 0.02j          # phase.impedance_matrix, archetype LV_4WIRE_PROXY_V1 (ohm/km, off-diagonal)
CABLE = {"overhead": ("LXS 4 x 95", 0.359, 0.1, 230.0),        # E-REDES DIT-C14-100/N: r_hot, x (ohm/km), Iz (A)
         "underground": ("LXAV 3x185+95", 0.21, 0.1, 375.0)}

def solve_lv(code, phase_shares, sn_mva, vk=4.0, vkr=None):
    res = q(f"select * from read_parquet('@lv/lv_geo_ptd_results.parquet') where ptd_code = '{code}'").iloc[0]
    s = q(f"select * from read_parquet('@lv/lv_geo_segments.parquet') where ptd_code = '{code}' order by node")
    t = q(f"select * from read_parquet('@lv/ptd_transformers.parquet') where ptd_code = '{code}'").iloc[0]
    vkr = t.vkr_percent if vkr is None else vkr
    n = int(s.node.max()) + 1
    parent = np.full(n, -1); length = np.zeros(n); par = np.ones(n); w = np.zeros(n)
    parent[s.node] = s.parent_node; length[s.node] = s.length_km; par[s.node] = s.refined_parallel; w[s.node] = s.load_weight
    kind = "overhead" if str(res.ctype).startswith("Aéreo") else "underground"
    _, r, x, imax = CABLE[kind]
    z = np.full((4, 4), MUTUAL); np.fill_diagonal(z, complex(r, x))
    base = 0.4 ** 2 / sn_mva
    zt = complex(vkr / 100 * base, math.sqrt(vk ** 2 - vkr ** 2) / 100 * base)
    p = res.load_kw / 1000; pv = res.pv_kw / 1000
    S = (p - pv + 1j * p * math.tan(math.acos(0.97))) * 1e6
    if w.sum() == 0: w[1:] = 1
    loads = np.outer(w / w.sum() * S, phase_shares)
    children = defaultdict(list)
    for k in range(1, n): children[parent[k]].append(k)
    order, dq = [0], deque([0])
    while dq:
        u = dq.popleft()
        for v in children[u]: order.append(v); dq.append(v)
    V = np.tile(SRC, (n, 1))
    for it in range(1, 101):
        I = np.zeros((n, 4), complex)
        I[:, :3] = np.conj(loads / (V[:, :3] - V[:, 3:4])); I[:, 3] = -I[:, :3].sum(1)
        for k in reversed(order[1:]): I[parent[k]] += I[k]
        Vn = V.copy(); Vn[0, :3] = SRC[:3] - zt * I[0, :3]; Vn[0, 3] = 0
        for k in order[1:]: Vn[k] = Vn[parent[k]] - z @ I[k] * length[k] / par[k]
        done = np.max(np.abs(Vn - V)) < 1e-6; V = Vn
        if done: break
    vpu = np.abs(V[:, :3] - V[:, 3:4]) / VLN
    loading = np.max(np.abs(I[1:, :3]).max(1) / (imax * par[1:])) * 100
    return dict(iterations=it, vmin=vpu.min(), vmax=vpu.max(), loading_pct=loading, neutral_A=np.abs(I[1:, 3]).max(),
                stored_vmin=res.geo_refined_vmin, stored_vmax=res.geo_refined_vmax, stored_loading_pct=res.geo_refined_loading_pct), (s, vpu)

# phase.lv_load_shares (A, B, C) and parameter.ptd_transformer_root_peak_design_scenario.design_sn_mva for this PTD
out, (s, vpu) = solve_lv("1106D1002700", phase_shares=[0.37, 0.34, 0.29], sn_mva=0.4)
pd.Series(out)

The sweep reproduces the stored minimum voltage and loading. Phase voltages along the network (the lowest of the three phases at each node):

In [ ]:
s = s.assign(vmin=vpu.min(1)[s.node])
fig, ax = plt.subplots(figsize=(6.5, 6))
lc = LineCollection([[(a, b_), (e, f)] for a, b_, e, f in zip(s.lon_from, s.lat_from, s.lon_to, s.lat_to)],
                    array=s.vmin.values, cmap="viridis", linewidths=1 + s.refined_parallel.values)
ax.add_collection(lc); ax.autoscale(); ax.set_aspect(1.3)
ax.plot(s.lon_from.iloc[0], s.lat_from.iloc[0], "ks", ms=7, label="PTD")
fig.colorbar(lc, ax=ax, label="lowest phase-neutral voltage (p.u.)"); ax.legend(); ax.set_title("1106D1002700, LV-GEO-A")
plt.show()

Without the two PTD-specific constants, balanced phase shares and the public transformer rating give a first approximation for any PTD. The database's phase shares range from 0.29 to 0.37, and this imbalance lowers the worst phase voltage, as the comparison shows:

In [ ]:
for code in ["0911D2009300", "1106D1002700"]:
    t = q(f"select sn_mva from read_parquet('@lv/ptd_transformers.parquet') where ptd_code = '{code}'").sn_mva.iloc[0]
    o, _ = solve_lv(code, phase_shares=[1/3, 1/3, 1/3], sn_mva=t)
    print(f"{code}: vmin {o['vmin']:.4f} (stored {o['stored_vmin']:.4f}), loading {o['loading_pct']:.1f}% (stored {o['stored_loading_pct']:.1f}%)")

**Citation.** SimPT-Power is a simulation database built from public data. It is *not* the operator's network or a digital twin: converged power flows show numerical consistency, not agreement with operator state estimates. Please cite the archived release (Zenodo DOI) and the data paper. Data licences follow the sources (E-REDES CC BY 4.0, OpenStreetMap-derived layers ODbL); see `SOURCE_LICENSES.md` in the release.